In [1]:
!pip install datasets==2.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.7.0
    Uninstalling datasets-4.7.0:
      Successfully uninstalled datasets-4.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
!pip install --upgrade datasets transformers av evaluate jiwer

import torch
import re
import json
import numpy as np
import evaluate
from datasets import load_dataset, Audio
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

  Using cached datasets-4.7.0-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.7.0-py3-none-any.whl (527 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 2.21.0
    Uninstalling datasets-2.21.0:
      Successfully uninstalled datasets-2.21.0


Тут ми завантажуємо датасет. 10% з нього виділяємо на валідацію, нормалізуємо.

In [3]:
print("    завантаження датасету    ")
dataset = load_dataset("speech-uk/opentts-mykyta", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
chars_to_remove_regex = r'[,?.!\-;:"“”—_]'
def remove_special_characters(batch):
    batch["transcription"] = batch["transcription"].lower()
    batch["transcription"] = re.sub(chars_to_remove_regex, '', batch["transcription"])
    return batch
dataset = dataset.map(remove_special_characters)
print("      готово     ")

    завантаження датасету    


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


      готово     


Генеруємо словник

In [4]:
def extract_all_chars(batch):
    all_text = " ".join(batch["transcription"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = dataset["train"].map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset["train"].column_names)
vocab_val = dataset["test"].map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset["test"].column_names)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_val["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
vocab_dict = {k: v+2 for k, v in vocab_dict.items()}
vocab_dict["<blank>"] = 0
vocab_dict["<unk>"] = 1
if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict[" "]
    del vocab_dict[" "]

with open('vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

Map:   0%|          | 0/5792 [00:00<?, ? examples/s]

Map:   0%|          | 0/644 [00:00<?, ? examples/s]

Перетворюємо звук на масиви числез через FetureExtractot та тексту на індекси через токанайзер

In [5]:
tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="<unk>", pad_token="<blank>", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=False)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = processor(text=batch["transcription"]).input_ids
    return batch

dataset_prepared = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names)

print("дані готові, ура ура")

дані готові, ура ура


In [6]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model, Wav2Vec2Config

class CustomSSLForCTC(nn.Module):
    def __init__(self, model_name, vocab_size, layer_index=-1):
        super().__init__()
        self.layer_index = layer_index
        config = Wav2Vec2Config.from_pretrained(model_name)
        config.output_hidden_states = True
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name, config=config)
        self.wav2vec2.feature_extractor._freeze_parameters()
        hidden_size = config.hidden_size
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_values, attention_mask=None, labels=None):
        outputs = self.wav2vec2(
            input_values=input_values,
            attention_mask=attention_mask
        )

        hidden_states = outputs.hidden_states
        selected_features = hidden_states[self.layer_index]
        logits = self.lm_head(selected_features)

        return logits

print("ініціалізація")
vocab_size = len(vocab_dict)
model = CustomSSLForCTC(
    model_name="facebook/wav2vec2-base",
    vocab_size=vocab_size,
    layer_index=-2
)

print(f"слой: {model.layer_index}, розмір голови: {vocab_size} классов.")

ініціалізація


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


слой: -2, розмір голови: 37 классов.


Оцінимо якість того, як ми розпізнаватимо мову. Для цього тут буде використовуватися **word error rate** та **character error rate**. WER буде рахувати відсоток слів, які модель вгадала неправильно. CER працює +- за таким ж самим принципом, але на рівні букв. Тобто, CER більше для точності

In [7]:
!pip install evaluate jiwer

import numpy as np
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

print("готово")

готово


так як хаггінг фейс в новій версії заблокували завантаження через пайтонівські скрипти, то всі репозиторії з common voice впали. Тому, аби навчити хоч на чомусь, ми використали 10% датасету OpenTTS, які зберегли ще на етапі препроцесингу

In [14]:
from transformers import TrainingArguments, Trainer, Wav2Vec2ForCTC
import evaluate
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)
model.freeze_feature_encoder()

training_args = TrainingArguments(
    output_dir="./wav2vec2-custom-uk",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="steps",
    num_train_epochs=3,
    fp16=True,
    save_steps=200,
    eval_steps=200,
    logging_steps=50,
    learning_rate=3e-4,
    weight_decay=0.005,
    warmup_steps=100,
    save_total_limit=2,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_prepared["train"],
    eval_dataset=dataset_prepared["test"],
)

trainer.train()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2172, training_loss=2.2974386904560182, metrics={'train_runtime': 3998.9256, 'train_samples_per_second': 4.345, 'train_steps_per_second': 0.543, 'total_flos': 3.9065252119986365e+18, 'train_loss': 2.2974386904560182, 'epoch': 3.0})

видно, що CER впав з 1.0 до 0.05, значить можемо зробити висновок, що модель навчилась розпізнавати українську мову

Тепер зробимо порівняння оригінального тексту та розпізнаного:

In [15]:
import torch
from datasets import load_dataset, Audio

sample = dataset_prepared["test"][15]
inputs = processor(sample["input_values"], return_tensors="pt", sampling_rate=16000)
inputs = {k: v.to("cuda") for k, v in inputs.items()}
model.eval()
with torch.no_grad():
    logits = model(inputs["input_values"]).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)[0]
print(f"оригінальний: {processor.decode(sample['labels'])}")
print(f"розпізнаний:  {transcription}")

import IPython.display as ipd
ipd.Audio(data=dataset["test"][15]["audio"]["array"], autoplay=False, rate=16000)

оригінальний: голос дзвоника це був наш умовлений сигнал
розпізнаний:  голоз звоника це був наж у мовлениь сигн
